# Train and package a UPAR 2027 Track 1 submission (EVA02-B/14)

Run this notebook from top to bottom in Kaggle with **Internet** and a **GPU** enabled. It first downloads the challenge training and validation images and annotations, then fine-tunes an ImageNet-22k-pretrained **EVA02-B/14** Vision Transformer on the 40 provided attributes. It saves `/kaggle/working/upar_track1_eva02_b14_best.pth` and packages `/kaggle/working/upar_track1_eva02_b14_submission.zip` for Codabench. Download the ZIP from Kaggle Outputs and upload it to the Track 1 submission page.

The download requires enough free disk space for the image archives and extracted files. Training 97,669 images can take substantial time. EVA02-B uses 257 tokens per image (14 x 14 patches) instead of DeiT III-B's 197, so it is about 1.3 times slower: on a Kaggle T4 or P100 one epoch takes roughly 15-20 minutes. Lower `BATCH_SIZE` to 32 if you run out of GPU memory. Re-running the download cells skips images already present. Only the provided train and validation splits are used.

**Pretrained weights.** `timm` model `eva02_base_patch14_448.mim_in22k_ft_in22k_in1k` (87.1M parameters, 88.7% ImageNet-1K top-1 at 448 x 448, 350 MB, MIT license). EVA02 is pretrained on ImageNet-22k with masked image modeling, using EVA-CLIP features as the reconstruction target, then fine-tuned on ImageNet-22k and ImageNet-1k by the authors (https://arxiv.org/abs/2303.11331). The challenge rules permit this generic pretraining. timm downloads the weights from the Hugging Face Hub (`timm/eva02_base_patch14_448.mim_in22k_ft_in22k_in1k`) the first time. timm is not part of the evaluation container (torch 2.4.1, torchvision), so the generated `run.py` defines the EVA02 network in plain PyTorch with the same parameter names, and the fine-tuned weights ship inside the ZIP.

## 1. Download challenge data

These cells are copied from `UPAR-Challenge-2027/kaggle_download_datasets.ipynb`. They download Market1501, PA-100K, and PETA and copy the challenge annotations into `/kaggle/working/upar_2027/data`. Source dataset licenses and usage restrictions apply.


In [ ]:
%pip -q install gdown tqdm requests timm

In [ ]:
import csv
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

if not (REPO / 'peta_file_mapping.txt').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
for split in ('train', 'val'):
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / 'data' / 'annotations' / f'{split}.csv', destination)
print('Data directory:', DATA)

In [ ]:
# The annotation paths determine exactly which images must be available.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        for row in list(csv.reader(handle))[1:]:
            relative = Path(row[0])
            needed[relative.parts[0]].add(relative)

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()  # Remove an interrupted download before retrying.
    return path.exists()

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}')
    print(f'{source}: {len(needed[source]):,} annotated images available')

print({source: len(paths) for source, paths in needed.items()})

## Market1501

In [ ]:
market_archive = DATA / 'market_1501.zip'
market_dir = DATA / 'Market1501'
if missing('Market1501'):
    if not check_zip(market_archive):
        result = gdown.download(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            output=str(market_archive), quiet=False, use_cookies=False, fuzzy=True
        )
        if not result or not zipfile.is_zipfile(market_archive):
            raise RuntimeError('Market1501 download failed or is not a ZIP file')
    with zipfile.ZipFile(market_archive) as archive:
        archive.extractall(DATA)
    extracted = DATA / 'Market-1501-v15.09.15'
    if extracted.exists():
        shutil.copytree(extracted, market_dir, dirs_exist_ok=True)
        shutil.rmtree(extracted)
require_complete('Market1501')
market_archive.unlink(missing_ok=True)

## PA-100K

In [ ]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
if missing('PA100k'):
    if not check_zip(pa_archive):
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
            output=str(pa_dir), quiet=False, use_cookies=False
        )
        if not zipfile.is_zipfile(pa_archive):
            raise RuntimeError('PA-100K data.zip was not downloaded')
    with zipfile.ZipFile(pa_archive) as archive:
        archive.extractall(pa_dir)
require_complete('PA100k')
pa_archive.unlink(missing_ok=True)

## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [ ]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
if missing('PETA'):
    if not check_zip(peta_archive):
        url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
        temporary = peta_archive.with_suffix('.zip.part')
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        bar.update(len(chunk))
        temporary.replace(peta_archive)
        if not zipfile.is_zipfile(peta_archive):
            raise RuntimeError('PETA download is not a ZIP file')
    with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
        mapping = list(csv.reader(handle))
    with zipfile.ZipFile(peta_archive) as archive:
        members = set(archive.namelist())
        for source, destination in tqdm(mapping, desc='Extracting PETA images'):
            target = DATA / destination
            if target.is_file():
                continue
            member = source.removeprefix('PETA/')
            if member not in members:
                raise FileNotFoundError(f'PETA archive is missing {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as original, target.open('wb') as output:
                shutil.copyfileobj(original, output)
require_complete('PETA')
peta_archive.unlink(missing_ok=True)

In [ ]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

## 2. Training configuration

The download cells above create `DATA_ROOT` at `/kaggle/working/upar_2027/data`. Edit training settings here as needed. The EVA02 checkpoint was fine-tuned at 448 x 448, but this notebook trains at `IMAGE_SIZE` x `IMAGE_SIZE` = 224 x 224 to keep training affordable: timm interpolates the absolute position embeddings to the 16 x 16 patch grid, which is also the grid that EVA02's rotary position embeddings were pretrained on. Pedestrian crops are resized (stretched) to this square. EVA02 expects inputs normalized with the CLIP mean and standard deviation. Set `USE_IMAGENET_WEIGHTS = False` to train from scratch if pretrained weights are unavailable.

In [ ]:
"""Kaggle training script for UPAR 2027 Track 1 with a timm EVA02-B/14 (EVA02).

Reads challenge data from /kaggle/working/upar_2027/data by default.
This file is also the source for the training cells in the accompanying notebook.
"""

import csv
import importlib.util
import math
import os
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


DATA_ROOT = Path(os.environ.get("UPAR_DATA_ROOT", "/kaggle/working/upar_2027/data"))
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
MODEL_TAG = "upar_track1_eva02_b14"
ARCHITECTURE = "eva02_base_patch14_448"  # timm EVA02
PRETRAINED_TAG = "mim_in22k_ft_in22k_in1k"  # ImageNet-22k MIM pretraining, ImageNet-22k then ImageNet-1k fine-tuning.
EPOCHS = 8
WARMUP_EPOCHS = 1
BATCH_SIZE = 64
NUM_WORKERS = 4
LEARNING_RATE = 1e-4  # Peak learning rate of the classifier head.
LAYER_DECAY = 0.75  # Each earlier transformer block gets this factor less learning rate.
WEIGHT_DECAY = 0.05
DROP_PATH = 0.1  # Stochastic depth rate.
USE_IMAGENET_WEIGHTS = True  # Needs Internet once (Hugging Face Hub), or a cached timm checkpoint.
IMAGE_SIZE = 224  # 16 x 16 patches of 14 pixels, the grid of EVA02's rotary position embeddings.
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
SEED = 42

## 3. Read annotations and images

The released training CSV header omits the image-column label; this parser handles both train and validation formats.

In [ ]:
def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attributes in {path}, got {len(names)}")
        paths, labels = [], []
        for row_number, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{row_number}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path} contains labels other than 0 and 1")
    return names, paths, labels


class AttributeDataset(Dataset):
    def __init__(self, root, paths, labels, transform):
        self.root, self.paths, self.labels, self.transform = root, paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.from_numpy(self.labels[index])




## 4. Model, metrics, and training

The model averages the patch tokens of the last block and predicts 40 independent attribute probabilities. Validation uses the provided validation split; the hidden test set is never accessed. The checkpoint with the best **Challenge_Avg** on validation is kept: the harmonic mean of label-based mean accuracy (mA) and instance-based F1 at the 0.5 threshold, computed per source dataset and averaged, like the leaderboard.

In [ ]:
def build_model(pretrained):
    model = timm.create_model(
        f"{ARCHITECTURE}.{PRETRAINED_TAG}", pretrained=pretrained, num_classes=40,
        img_size=IMAGE_SIZE, drop_path_rate=DROP_PATH,
    )
    nn.init.zeros_(model.head.weight)
    nn.init.zeros_(model.head.bias)
    return model


def parameter_groups(model):
    """AdamW groups with layer-wise learning-rate decay and no decay on norms/biases."""
    depth = len(model.blocks)

    def layer_id(name):
        if name.startswith(("cls_token", "pos_embed", "patch_embed")):
            return 0
        if name.startswith("blocks."):
            return int(name.split(".")[1]) + 1
        return depth + 1  # fc_norm and classifier head.

    groups = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        no_decay = parameter.ndim == 1 or name in ("cls_token", "pos_embed")
        lid = layer_id(name)
        key = (lid, no_decay)
        if key not in groups:
            groups[key] = {
                "params": [],
                "weight_decay": 0.0 if no_decay else WEIGHT_DECAY,
                "lr_scale": LAYER_DECAY ** (depth + 1 - lid),
            }
        groups[key]["params"].append(parameter)
    return list(groups.values())


def challenge_metrics(probabilities, labels, sources):
    """mA, instance F1 and their harmonic mean, averaged over source datasets."""
    eps = 1e-20
    results = []
    for source in sorted(set(sources)):
        rows = np.asarray([s == source for s in sources])
        pred = probabilities[rows] > 0.5
        gt = labels[rows] > 0.5
        tp = (pred & gt).sum(0)
        tn = (~pred & ~gt).sum(0)
        positives, negatives = gt.sum(0), (~gt).sum(0)
        label_ma = (tp / (positives + eps) + tn / (negatives + eps)) / 2
        intersection = (pred & gt).sum(1)
        precision = (intersection / (pred.sum(1) + eps)).mean()
        recall = (intersection / (gt.sum(1) + eps)).mean()
        # As in the original evaluation script: F1 of the image-averaged precision and recall.
        ma, f1 = label_ma.mean(), 2 * precision * recall / (precision + recall + eps)
        results.append((ma, f1, 2 * ma * f1 / (ma + f1 + eps)))
    ma, f1, avg = np.mean(results, axis=0)
    return {"mA": ma, "Instance_F1": f1, "Challenge_Avg": avg}

In [ ]:
def main():
    if not (DATA_ROOT / "annotations/train.csv").is_file():
        raise FileNotFoundError(
            f"Missing {DATA_ROOT / 'annotations/train.csv'}. Run the download notebook "
            "in this Kaggle session or attach its saved output and set UPAR_DATA_ROOT."
        )
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True

    names, train_paths, train_labels = read_annotations(DATA_ROOT / "annotations/train.csv")
    val_names, val_paths, val_labels = read_annotations(DATA_ROOT / "annotations/val.csv")
    if names != val_names:
        raise ValueError("Train and validation attribute orders differ")
    for relative in train_paths[:10] + val_paths[:10]:
        if not (DATA_ROOT / relative).is_file():
            raise FileNotFoundError(f"Missing image {DATA_ROOT / relative}; finish downloading the images")
    val_sources = [Path(path).parts[0] for path in val_paths]
    print(f"Loaded {len(train_paths):,} train and {len(val_paths):,} validation images")

    normalize = transforms.Normalize(MEAN, STD)
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.Pad(10, padding_mode="edge"),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        normalize,
        transforms.RandomErasing(p=0.25, value="random"),
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), normalize,
    ])
    train_loader = DataLoader(
        AttributeDataset(DATA_ROOT, train_paths, train_labels, train_transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(), drop_last=True,
    )
    val_loader = DataLoader(
        AttributeDataset(DATA_ROOT, val_paths, val_labels, val_transform),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    model = build_model(USE_IMAGENET_WEIGHTS).to(device)
    print(f"{ARCHITECTURE}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    positives = train_labels.sum(axis=0)
    negative_to_positive = (len(train_labels) - positives) / np.maximum(positives, 1)
    pos_weight = torch.tensor(np.clip(negative_to_positive, 1, 20), dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(parameter_groups(model), lr=LEARNING_RATE)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = WARMUP_EPOCHS * len(train_loader)

    def learning_rate(step):
        if step < warmup_steps:
            return LEARNING_RATE * (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * progress))

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    checkpoint_path = OUTPUT_ROOT / f"{MODEL_TAG}_best.pth"
    best_score = -1.0
    step = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_total = 0.0
        for images, labels in train_loader:
            for group in optimizer.param_groups:
                group["lr"] = learning_rate(step) * group["lr_scale"]
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_total += loss.item() * len(images)
            step += 1

        model.eval()
        val_total = 0.0
        val_probabilities = []
        with torch.inference_mode():
            for images, labels in val_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                    logits = model(images)
                val_total += criterion(logits.float(), labels).item() * len(images)
                val_probabilities.append(logits.float().sigmoid().cpu().numpy())
        metrics = challenge_metrics(np.concatenate(val_probabilities), val_labels, val_sources)
        train_loss = train_total / (len(train_loader) * BATCH_SIZE)
        val_loss = val_total / len(val_paths)
        print(
            f"Epoch {epoch}/{EPOCHS}: train BCE={train_loss:.4f}, val BCE={val_loss:.4f}, "
            f"mA={metrics['mA']:.4f}, Inst_F1={metrics['Instance_F1']:.4f}, "
            f"Challenge_Avg={metrics['Challenge_Avg']:.4f}"
        )
        if metrics["Challenge_Avg"] > best_score:
            best_score = metrics["Challenge_Avg"]
            torch.save({
                # Stored in float16 to halve the ZIP size; load_state_dict casts back to float32.
                "state_dict": {
                    key: (value.detach().half() if value.is_floating_point() else value.detach()).cpu()
                    for key, value in model.state_dict().items()
                },
                "architecture": ARCHITECTURE,
                "num_heads": model.blocks[0].attn.num_heads,
                "attribute_names": names,
                "image_size": IMAGE_SIZE,
                "epoch": epoch,
                "val_loss": val_loss,
                "val_metrics": {key: float(value) for key, value in metrics.items()},
            }, checkpoint_path)
            print(f"Saved {checkpoint_path}")

    package_submission(checkpoint_path, [DATA_ROOT / path for path in val_paths[:4]], names)

## 5. Build the inference package

The generated `run.py` exposes the Track 1 API: `load_model()`, `predict_image(sample)` and `predict_batch(samples)`. It defines EVA02 in plain PyTorch (timm is not in the evaluation image), loads the checkpoint, averages the predictions of each image and its horizontal flip, and returns 40 probabilities in the order of `sample["attribute_names"]`. `metadata.yaml` is the unchanged file from `examples/task1/sample_code_submission`. The smoke test also checks that the plain-PyTorch network gives the same logits as the timm model.

In [ ]:
RUN_PY = '''"""UPAR 2027 Track 1 inference entry point (EVA02-B/14).

timm is not installed in the evaluation image, so EVA02 is defined here in plain PyTorch.
Parameter names match timm's Eva, so the fine-tuned state dict loads strictly.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision import transforms

HERE = Path(__file__).resolve().parent
CHECKPOINT = HERE / "assets/upar_track1_eva02_b14_best.pth"
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
BATCH_SIZE = 64
MODEL = None
ATTRIBUTE_NAMES = None
TRANSFORM = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, dim):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


def rotary_embedding(grid_size, head_dim, ref_grid_size=16, temperature=10000.0):
    """timm RotaryEmbeddingCat (in_pixels=False): sin and cos of 2D patch positions, concatenated."""
    num_bands = head_dim // 4
    bands = 1.0 / temperature ** (torch.arange(num_bands, dtype=torch.float32) / num_bands)
    # EVA02 scales patch coordinates to its 224 / 14 = 16 x 16 pretraining grid.
    t = torch.arange(grid_size, dtype=torch.float32) / grid_size * ref_grid_size
    grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1).unsqueeze(-1)
    positions = (grid * bands).reshape(grid_size * grid_size, -1)
    return torch.cat([positions.sin().repeat_interleave(2, -1), positions.cos().repeat_interleave(2, -1)], -1)


def apply_rotary(x, embedding):
    sin, cos = embedding.chunk(2, -1)
    rotated = torch.stack([-x[..., 1::2], x[..., ::2]], -1).reshape(x.shape)  # (x0, x1) -> (-x1, x0)
    return x * cos + rotated * sin


class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, rope):
        b, n, c = x.shape
        q, k, v = (
            projection(x).reshape(b, n, self.num_heads, -1).transpose(1, 2)
            for projection in (self.q_proj, self.k_proj, self.v_proj)
        )
        # Rotary position embeddings rotate the patch tokens only, not the class token.
        q = torch.cat([q[:, :, :1], apply_rotary(q[:, :, 1:], rope)], dim=2).type_as(v)
        k = torch.cat([k[:, :, :1], apply_rotary(k[:, :, 1:], rope)], dim=2).type_as(v)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1_g = nn.Linear(dim, hidden_dim)
        self.fc1_x = nn.Linear(dim, hidden_dim)
        self.act = nn.SiLU()
        self.norm = nn.LayerNorm(hidden_dim, eps=1e-6)
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.norm(self.act(self.fc1_g(x)) * self.fc1_x(x)))


class Block(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = SwiGLU(dim, hidden_dim)

    def forward(self, x, rope):
        x = x + self.attn(self.norm1(x), rope)
        return x + self.mlp(self.norm2(x))


class EVA02(nn.Module):
    """timm Eva for EVA02: RoPE, SwiGLU with sub-LayerNorm, average pooling of patch tokens."""

    def __init__(self, patch_size, num_patches, dim, depth, num_heads, hidden_dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, num_heads, hidden_dim) for _ in range(depth)])
        self.fc_norm = nn.LayerNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, num_classes)
        self.register_buffer("rope", rotary_embedding(math.isqrt(num_patches), dim // num_heads), persistent=False)

    def forward(self, x):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(len(x), -1, -1), x], dim=1) + self.pos_embed
        for block in self.blocks:
            x = block(x, self.rope)
        return self.head(self.fc_norm(x[:, 1:].mean(dim=1)))


def load_model():
    global MODEL, ATTRIBUTE_NAMES, TRANSFORM
    checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    ATTRIBUTE_NAMES = checkpoint["attribute_names"]
    size = int(checkpoint["image_size"])
    TRANSFORM = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    state = checkpoint["state_dict"]
    model = EVA02(
        patch_size=state["patch_embed.proj.weight"].shape[-1],
        num_patches=state["pos_embed"].shape[1] - 1,  # pos_embed includes the class token.
        dim=state["pos_embed"].shape[2],
        depth=len({key.split(".")[1] for key in state if key.startswith("blocks.")}),
        num_heads=int(checkpoint["num_heads"]),
        hidden_dim=state["blocks.0.mlp.fc1_g.weight"].shape[0],
        num_classes=state["head.weight"].shape[0],
    )
    model.load_state_dict(state)
    MODEL = model.float().to(DEVICE).eval()


def predict_batch(samples):
    if MODEL is None:
        load_model()
    images = []
    for sample in samples:
        with Image.open(sample["image_path"]) as image:
            images.append(TRANSFORM(image.convert("RGB")))
    batch = torch.stack(images).to(DEVICE)
    with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        logits = MODEL(batch).float() + MODEL(torch.flip(batch, dims=[3])).float()
    probabilities = (logits / 2).sigmoid().clamp(0.0, 1.0).cpu().numpy()
    probabilities = np.nan_to_num(probabilities, nan=0.0)
    rows = []
    for sample, row in zip(samples, probabilities):
        order = [ATTRIBUTE_NAMES.index(name) for name in sample["attribute_names"]]
        rows.append(row[order].astype(float).tolist())
    return rows


def predict_image(sample):
    return predict_batch([sample])[0]
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls predict_image() / predict_batch(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(checkpoint_path, sample_images, names):
    folder = OUTPUT_ROOT / f"{MODEL_TAG}_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    shutil.copy2(checkpoint_path, assets / checkpoint_path.name)
    archive_path = OUTPUT_ROOT / f"{MODEL_TAG}_submission.zip"
    expected = {"run.py", "metadata.yaml", f"assets/{checkpoint_path.name}"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("upar_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            module.load_model()
            samples = [
                {"index": i, "image": str(path), "image_path": str(path), "attribute_names": names}
                for i, path in enumerate(sample_images)
            ]
            batch = np.asarray(module.predict_batch(samples))
            single = np.asarray(module.predict_image(samples[0]))
            assert batch.shape == (len(samples), 40) and single.shape == (40,)
            assert np.isfinite(batch).all() and ((batch >= 0) & (batch <= 1)).all()
            print("Packaged inference smoke test passed")
            # run.py re-implements EVA02; it must match the timm model exactly.
            reference = build_model(False).to(module.DEVICE).eval()
            reference.load_state_dict(module.MODEL.state_dict())
            inputs = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=module.DEVICE)
            with torch.inference_mode():
                difference = (reference(inputs) - module.MODEL(inputs)).abs().max().item()
            assert difference < 1e-3, f"run.py and timm outputs differ by {difference}"
            print(f"run.py matches timm (max logit difference {difference:.1e})")
    print("Checkpoint:", checkpoint_path)
    print(f"ZIP size: {archive_path.stat().st_size / 1e6:.0f} MB")
    print("Upload this ZIP to Codabench:", archive_path)

## 6. Run training and create the ZIP

After this cell finishes, find the `.pth` checkpoint and submission ZIP under `/kaggle/working`. Validation Challenge_Avg selects the checkpoint; the leaderboard ranks submissions by Challenge_Avg on the hidden test domain, which usually scores lower than the provided validation split.

In [ ]:
main()